# Nearest-neighbour baseline — does the river graph earn its place at all?

The distance control showed the gain does not need the true upstream topology. The direct
consequence, which the paper never tests, is that the graph may be unnecessary **scaffolding**: if
the useful quantity is "recent discharge from nearby gauges," then simply averaging the *k*
geographically nearest basins should work as well, with no river network, no edge inference, and no
area ratio or elevation rule.

This is the baseline a reviewer names first once the title says "Proximity, Not Topology." If it
matches the graph-derived input, the honest framing is that the network is a convenience, not a
requirement — which is a cleaner and more transferable claim than the paper currently makes.

Pre-registration: `experiments/topology_ablation/preregistration_knn_baseline.md`.

**Runtime → Change runtime type → T4 GPU → Run all.** ~2 h.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config (SEEDS = [11, 13, 17], k in {2, 4})

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # blank -> auto-detect
SEEDS=[11,13,17]
K_LIST=[2,4]  # neighbours to average
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; break
if not DRIVE_CAMELS_PATH or not os.path.isdir(DRIVE_CAMELS_PATH):
    raise RuntimeError(f'CAMELS not found. Tried: {AUTO}')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydro_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('CAMELS:', DRIVE_CAMELS_PATH); print('RUNS  :', DRIVE_RUNS)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(f'{DRIVE_RUNS}/topology_ablation/component0', exist_ok=True)
print('datasets ->', os.path.realpath(RD)); print('runs     ->', os.path.realpath(RR))

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Build the k-nearest-neighbour features (no graph)

For each basin, take the *k* geographically nearest **other** basins by great-circle gauge distance,
and area-weight their lagged discharge exactly as Eq. 4 does. The river graph is not consulted: no
edge rule, no area ratio, no elevation. Every basin gets a feature, including headwaters — which the
graph-based input cannot do, and which is itself informative.

In [ ]:
%cd {REPO_DIR}
import pickle, numpy as np, pandas as pd
from pathlib import Path
FEAT='experiments/topology_ablation/features'
P1=Path('topology_analysis/phase1_network_discovery/outputs')
TOPO_TXT='datasets/camels_us/camels_attributes_v2.0/camels_topo.txt'
def named_ok(p, n_expected=183):
    # must exist, have a 'date'-named index, AND cover every basin (NH KeyErrors on a missing one)
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb'))
    if d[next(iter(d))].index.name!='date': return False
    if len(d)<n_expected:
        print(f'  [rebuild] {os.path.basename(p)} covers {len(d)}/{n_expected} basins'); return False
    return True

def haversine(a1,o1,a2,o2):
    R=6371.0; p1,p2=np.radians(a1),np.radians(a2)
    dp=np.radians(a2-a1); dl=np.radians(o2-o1)
    h=np.sin(dp/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(h))

def build_knn(k):
    out=f'{FEAT}/upstream_q_knn{k}_component0_lag1.p'
    if named_ok(out): print(f'k={k}: present, skipping'); return out
    from neuralhydrology.datasetzoo.camelsus import load_camels_us_discharge
    basins=[l.strip() for l in open(P1/'component0_basins.txt') if l.strip()]
    topo=pd.read_csv(TOPO_TXT,sep=';',dtype={'gauge_id':str}).set_index('gauge_id')
    lat={b:float(topo.loc[b,'gauge_lat']) for b in basins}
    lon={b:float(topo.loc[b,'gauge_lon']) for b in basins}
    area={b:float(topo.loc[b,'area_gages2']) for b in basins}
    q={}
    for b in basins:
        try: q[b]=load_camels_us_discharge(Path('datasets/camels_us'),b,area[b])
        except Exception: q[b]=None
    feats={}; lens=[]
    # exclude each basin's TRUE parents so this arm is geography-only, zero topology overlap
    E=pd.read_csv(P1/'component0_edges.csv',dtype={'parent_id':str,'child_id':str})
    tp={}
    for pp,cc in zip(E.parent_id,E.child_id): tp.setdefault(cc,set()).add(pp)
    for c_ in basins:
        forb=tp.get(c_,set())|{c_}
        others=[o for o in basins if o not in forb]
        others.sort(key=lambda o: haversine(lat[c_],lon[c_],lat[o],lon[o]))
        nb=others[:k]
        lens += [haversine(lat[c_],lon[c_],lat[o],lon[o]) for o in nb]
        idx=None
        for o in nb:
            if q.get(o) is not None: idx=q[o].index; break
        if idx is None: continue
        agg=pd.Series(0.0,index=idx); wsum=0.0
        for o in nb:
            if q.get(o) is None: continue
            agg=agg.add((q[o].reindex(idx)*area[o]).fillna(0.0),fill_value=0.0); wsum+=area[o]
        if wsum<=0: continue
        s=(agg/wsum).shift(1).fillna(0.0)
        df=pd.DataFrame({'upstream_q':s.values},
                        index=pd.DatetimeIndex(idx,name='date'))
        feats[c_]=df
    pickle.dump(feats,open(out,'wb'))
    print(f'k={k}: {len(feats)} basins | mean neighbour distance {np.mean(lens):.1f} km')
    return out
for k in K_LIST: build_knn(k)
for k in K_LIST: print(f'  knn{k} ->', named_ok(f'{FEAT}/upstream_q_knn{k}_component0_lag1.p'))

## Cell 8 — Train the k-NN conditions

In [ ]:
%cd {REPO_DIR}
B=f'{REPO_DIR}/runs/topology_ablation/component0'
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
for k in K_LIST:
    for s in SEEDS:
        cond=f'L_upQknn{k}'
        if done(cond,s): print(f'{cond} seed {s}: done'); continue
        print(f'=== training {cond} seed {s} ===')
        !python experiments/topology_ablation/run_upstream_feature.py \
            --network component0 --seed {s} --device cuda:0 --epochs 30 \
            --feature-file experiments/topology_ablation/features/upstream_q_knn{k}_component0_lag1.p \
            --cond-name {cond}

## Cell 9 — Verdict: does the graph add anything over plain geography?

Compared on the **forward-connected basins** so the contrast with the graph conditions is paired on
the same set. The all-183 column is also printed, since k-NN is defined everywhere and the
graph input is not.

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np, pickle, glob
from scipy.stats import wilcoxon
FEAT='experiments/topology_ablation/features'
B=f'{REPO_DIR}/runs/topology_ablation/component0'
def named_ok(p, n_expected=183):
    # must exist, have a 'date'-named index, AND cover every basin (NH KeyErrors on a missing one)
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb'))
    if d[next(iter(d))].index.name!='date': return False
    if len(d)<n_expected:
        print(f'  [rebuild] {os.path.basename(p)} covers {len(d)}/{n_expected} basins'); return False
    return True
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
def nse(cond,s):
    p=f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
_f=pickle.load(open(f'{FEAT}/upstream_q_component0_lag1.p','rb'))
CONN=sorted([b for b,v in _f.items() if float(np.nanmax(np.abs(v.values)))>0])
def paired(cond,s,ref='L',basins=None):
    A=nse(cond,s); L=nse(ref,s)
    if A is None or L is None: return None
    bs=[b for b in (basins or CONN) if b in A.index and b in L.index]
    return (A[bs]-L[bs]).values
print('helpers ready | connected basins:', len(CONN))

In [ ]:
ALLB=sorted(set(nse('L',SEEDS[0]).index))
print('| condition | seed | Δ connected (n=150) | Δ all basins (n=183) |')
print('|---|---|---|---|')
rows={}
for cond in ['L_upQ','L_upQdistctrl']+[f'L_upQknn{k}' for k in K_LIST]:
    per=[]
    for s in SEEDS:
        dc=paired(cond,s); da=paired(cond,s,basins=ALLB)
        if dc is None: continue
        per.append(np.median(dc))
        print(f'| {cond} | {s} | {np.median(dc):+.4f} | {np.median(da):+.4f} |')
    if per: rows[cond]=per
print()
for cond,per in rows.items():
    print(f'{cond:16s} cross-seed mean Δ (connected) = {np.mean(per):+.4f}   per-seed {[f"{x:+.3f}" for x in per]}')

if 'L_upQ' in rows and f'L_upQknn{K_LIST[0]}' in rows:
    g=np.mean(rows['L_upQ']); best=max(np.mean(rows[f'L_upQknn{k}']) for k in K_LIST if f'L_upQknn{k}' in rows)
    print('\n=== PRE-REGISTERED VERDICT ===')
    if best >= g-0.005:
        print(f'  THE GRAPH IS SCAFFOLDING. Plain k-NN ({best:+.4f}) matches the river-graph input ({g:+.4f}).')
        print('  Report k-NN as the primary baseline; the network is a convenience, not a requirement.')
    elif best < g-0.005:
        print(f'  THE GRAPH ADDS SOMETHING. Graph {g:+.4f} > k-NN {best:+.4f} by {g-best:+.4f}.')
        print('  Upstream selection beats plain geography: a real, reportable residual for topology.')

## Cell 10 — Persistence check

In [ ]:
print('=== persistence (in Drive?) ===')
for k in K_LIST:
    for s in SEEDS:
        dp=f'{DRIVE_RUNS}/topology_ablation/component0/L_upQknn{k}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
        print(f'  L_upQknn{k} seed {s}: {os.path.isfile(dp)}')

## Done

Report **Cell 9** back. Either outcome is publishable: if k-NN matches, the paper's claim simplifies
to "nearby discharge, graph optional", which is stronger and more transferable. If the graph wins,
that residual is the first positive evidence for topology in the whole study.